# 30. Gold Layer — Slowly Changing Dimensions (SCD Type 2) for Title Dimension
**Layer:** Medallion Architecture -> **Gold Layer** (`playback_lakehouse.gold.dim_title`)  
**Pattern:** Single-Pass Delta `MERGE` with UNION ALL & Hash-Based Change Detection

---

## Purpose
This notebook processes clean catalog snapshots from the Silver layer and upserts them into the Gold dimension table using the **SCD Type 2 (Slowly Changing Dimensions)** pattern. It maintains a fully traceable historical timeline of movie and episode metadata mutations (such as genre shifts or title renames) over time.

---

## Architectural Mechanics

### 1. Hash-Based Change Detection (`attr_hash`)
Instead of comparing dozens of descriptive columns row-by-row, the staging layer generates a deterministic `sha2(concat_ws(...))` signature over all tracking attributes. If a movie's attributes change between snapshots, its `attr_hash` shifts, automatically triggering a history split.

### 2. Single-Pass SCD2 via `UNION ALL` Staging
To avoid expensive double-scan mutations, the `USING` block combines two logical paths into a single source dataset (`staged`):
* **Standard Stream:** Ingests the incoming snapshot records with their native business keys (`mergeKey = title_id`) to identify matches and handle basic inserts.
* **History Splitter:** Ingests records that *already exist* in Gold as active versions but have a conflicting `attr_hash`. By driving them with a `NULL` key (`mergeKey = CAST(NULL AS BIGINT)`), it guarantees they trigger the `WHEN NOT MATCHED` block, forcing the insertion of the new historical version.

---

## MERGE Logic Branches
* **`WHEN MATCHED` (Close Old Version):** If the `title_id` matches an active record (`is_current = true`) but the `attr_hash` differs, the existing record is soft-closed by setting `is_current = false` and updating `t.valid_to = DATE'{snap}'`.
* **`WHEN NOT MATCHED` (Open New Version):** Inserts completely new catalog entries or the newly updated versions generated by Copy B, initializing them with `is_current = true` and `valid_from = DATE'{snap}'`.

In [0]:
from conf.config import GOLD_DIM_TITLE, SILVER_DIM_TITLE, SNAP_V1, SNAP_V2, HIGH_DATE
from pyspark.sql import functions as F, Window

In [0]:
from pyspark.sql import functions as F

def scd2_merge(snap: str):
    """Atomic SCD2 upsert of one catalog snapshot into gold.dim_title."""

    # Staging view: the snapshot's clean rows + attribute hash for change detection
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW stg_dim_title_{snap.replace('-','')} AS
        SELECT title_id, title_name, genres, content_type, release_year, runtime_min,
               _snapshot_date,
               sha2(concat_ws('||',
                   coalesce(cast(title_id as string), ''),
                   coalesce(title_name, ''),
                   coalesce(array_join(genres, ','), ''),
                   coalesce(content_type, ''),
                   coalesce(cast(release_year as string), ''),
                   coalesce(cast(runtime_min as string), '')
               ), 256) AS attr_hash
        FROM {SILVER_DIM_TITLE}
        WHERE _snapshot_date = '{snap}'
    """)

    spark.sql(f"""
        MERGE INTO {GOLD_DIM_TITLE} AS t
        USING (
            SELECT CAST(s.title_id AS BIGINT) AS mergeKey, s.*
            FROM stg_dim_title_{snap.replace('-','')} s

            UNION ALL

            SELECT CAST(NULL AS BIGINT) AS mergeKey, s.*
            FROM stg_dim_title_{snap.replace('-','')} s
            JOIN {GOLD_DIM_TITLE} c
              ON  s.title_id  = c.title_id
              AND c.is_current = true
              AND s.attr_hash <> c.attr_hash
        ) AS staged
        ON t.title_id = staged.mergeKey AND t.is_current = true

        WHEN MATCHED AND t.attr_hash <> staged.attr_hash THEN
            UPDATE SET t.is_current = false,
                       t.valid_to   = DATE'{snap}'

        WHEN NOT MATCHED THEN
            INSERT (title_sk, attr_hash, title_id, title_name, genres, content_type,
                    release_year, runtime_min, valid_from, valid_to, is_current)
            VALUES (xxhash64(staged.title_id, staged._snapshot_date), staged.attr_hash,
                    staged.title_id, staged.title_name, staged.genres, staged.content_type,
                    staged.release_year, staged.runtime_min,
                    DATE'{snap}', DATE'9999-12-31', true)
    """)
    print(f"SCD2 merge for {snap} done.")


silver_snaps = {r[0] for r in spark.table(SILVER_DIM_TITLE)
                    .select("_snapshot_date").distinct().collect()}
merged_snaps = {str(r[0]) for r in spark.table(GOLD_DIM_TITLE)
                    .select("valid_from").distinct().collect()}

new_snaps = sorted(silver_snaps - merged_snaps)
if not new_snaps:
    print("No new snapshots to merge.")
for snap in new_snaps:
    scd2_merge(snap)